# Chapter 27 — Difference Equations and Markov Chains

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch27_markov_chains.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 27.1 First-order difference equations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def iterate(a, b, x0, periods):
    """Iterate x[t+1] = a*x[t] + b and return the whole path."""
    x = np.empty(periods + 1)
    x[0] = x0
    for t in range(periods):
        x[t + 1] = a * x[t] + b
    return x


a, b, x0 = 0.8, 2.0, 0.0
path = iterate(a, b, x0, 10)
x_star = b / (1 - a)
formula = x_star + a ** np.arange(11) * (x0 - x_star)
print(f"steady state x* = {x_star}")
print("iteration:", path.round(3))
print("formula:  ", formula.round(3))
print("same?", np.allclose(path, formula))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharex=True)
for ax, a_ in zip(axes, [0.8, -0.8, 1.1, -1.1]):
    b_ = 2.0 * (1 - a_)                       # keeps the steady state at 2
    x = iterate(a_, b_, 0.0, 20)
    ax.plot(range(21), x, marker="o", ms=3, color="#1f4e79", lw=1.2)
    ax.axhline(2.0, color="#c55a11", ls="--", lw=1)
    ax.set_title(f"a = {a_}", loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel("t")
axes[0].set_ylabel("$x_t$")
fig.suptitle("Stable and unstable paths of $x_{t+1} = a x_t + b$ "
             "(dashed line: steady state)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

### An economic example: the cobweb

In [ ]:
def cobweb(alpha, beta, gamma, delta, p0, periods=12):
    a_ = -delta / beta
    b_ = (alpha - gamma) / beta
    return iterate(a_, b_, p0, periods), b_ / (1 - a_)


for beta, delta in [(1.0, 0.7), (1.0, 1.2)]:
    prices, p_star = cobweb(alpha=100, beta=beta, gamma=10, delta=delta, p0=30)
    print(f"beta={beta}, delta={delta}: |a| = {delta / beta:.1f}, "
          f"P* = {p_star:.1f}; prices: {prices[:8].round(1)}")

## 27.2 Unemployment as a difference equation

In [ ]:
for s, f in [(0.01, 0.15), (0.05, 0.75)]:
    print(f"s={s}, f={f}: u* = {s / (s + f):.2%}, "
          f"mean duration of unemployment = {1 / f:.1f} quarters, "
          f"a = 1 - s - f = {1 - s - f:.2f}")

## 27.3 The PLFS urban panel

In [ ]:
from pathlib import Path

from indiadata import labour_status, layout_names, read_plfs

PLFS = Path("data/plfs")
ids = pd.DataFrame({"perv1": layout_names("perv1")[:17],
                    "perrv": layout_names("perrv")[:17]})
print("identical:", (ids["perv1"] == ids["perrv"]).all())
print(", ".join(ids["perv1"]))

In [ ]:
HH = ["st", "strm", "sstrm", "ss", "mfsu", "seg", "sss", "ssu"]
cols = ["seg", "sss", "ssu", "srl", "sex", "age", "acws"]

v1 = read_plfs("perv1", columns=cols)
v1 = v1[v1["sec"] == 2]                            # urban only
rv = read_plfs("perrv", columns=cols)               # revisits: all urban
print(f"first visits (urban): {len(v1):,} persons;  revisits: {len(rv):,}")

people = pd.concat([v1, rv], ignore_index=True)
del v1, rv
people = people[HH + ["qtr", "visit", "srl", "sex", "age", "acws",
                      "nss", "nsc", "mult", "weight"]]
people["q"] = people["qtr"].astype(str).str[1].astype(int)      # Q1..Q4 -> 1..4
people["v"] = people["visit"].astype(str).str[1].astype(int)    # V1..V4 -> 1..4
pd.crosstab(people["qtr"], people["visit"])

In [ ]:
people["wq"] = np.where(people["nss"] == people["nsc"],
                        people["mult"] / 100, people["mult"] / 200)
people["status"] = labour_status(people["acws"], weekly=True)
people = people.drop(columns=["nss", "nsc", "mult", "acws"])

# Is (household, person, quarter, visit) a unique key?
key = HH + ["srl", "q", "v"]
print("duplicate keys:", people.duplicated(key).sum())
print(f"memory: {people.memory_usage(deep=True).sum() / 1e6:.0f} MB")

### The actual unemployment rate, quarter by quarter

In [ ]:
from indiadata import labour_rates

adults = people[people["age"] >= 15]
ours = pd.DataFrame({
    sex: adults[adults["sex"] == code].groupby("qtr", observed=True)
                .apply(lambda d: labour_rates(d, "status", "wq")["UR"])
    for sex, code in [("Male", 1), ("Female", 2)]})
ours["Person"] = adults.groupby("qtr", observed=True).apply(
    lambda d: labour_rates(d, "status", "wq")["UR"])

published = pd.DataFrame(            # MoSPI, PLFS Quarterly Bulletin QB-23, Statement 3
    {"Male": [6.0, 5.8, 6.1, 5.8], "Female": [8.6, 8.6, 8.5, 9.0],
     "Person": [6.6, 6.5, 6.7, 6.6]},
    index=["Q1 (Jul-Sep 2023)", "Q2 (Oct-Dec 2023)",
           "Q3 (Jan-Mar 2024)", "Q4 (Apr-Jun 2024)"])
ours.index = published.index
pd.concat({"ours": ours.round(1), "MoSPI": published}, axis=1)

In [ ]:
actual_ur = ours["Person"].mean()
print(f"Average urban CWS unemployment rate, 15+, 2023-24 (ours): {actual_ur:.2f}%")
diff = (ours.round(1) - published).abs()
print(f"Cells that differ from the bulletin: {(diff > 0).sum().sum()} of 12; "
      f"largest difference {diff.max().max():.1f} point")

## 27.4 Linking visits into a panel

In [ ]:
later = people[key + ["status", "sex", "age"]].copy()
later["q"] -= 1                                 # so it lines up with the
later["v"] -= 1                                 # earlier visit's q and v
pairs = people.merge(later, on=key, suffixes=("", "_next"))
print(f"{len(pairs):,} person-pairs of consecutive visits")
print(pairs.groupby(["q", "v"]).size().rename("pairs").to_frame().T)

In [ ]:
hh_next = later[HH + ["q", "v"]].drop_duplicates()
eligible = people.merge(hh_next, on=HH + ["q", "v"])
match_rate = len(pairs) / len(eligible)
print(f"persons in revisited households: {len(eligible):,}; "
      f"matched: {len(pairs):,}; match rate {match_rate:.1%}")

same_sex = (pairs["sex"] == pairs["sex_next"]).fillna(False).mean()  # blank = mismatch
age_gap = (pairs["age_next"] - pairs["age"]).value_counts()
print(f"same sex: {same_sex:.2%}")
print("age at later visit minus age at earlier visit:", age_gap.to_dict())

# households: first visits in Q1-Q3 that were revisited next quarter
first = people[(people["v"] == 1) & (people["q"] <= 3)][HH + ["q", "v"]].drop_duplicates()
found = first.merge(hh_next, on=HH + ["q", "v"])
print(f"first-visit households (Q1-Q3) found at the next visit: "
      f"{len(found):,} of {len(first):,} ({len(found) / len(first):.1%})")
del eligible, hh_next, first, found

## 27.5 Transition matrices

In [ ]:
STATES = ["E", "U", "N"]
code = {"employed": "E", "unemployed": "U", "out of LF": "N"}
pairs = pairs[pairs["age"] >= 15].copy()
pairs["from"] = pairs["status"].map(code)
pairs["to"] = pairs["status_next"].map(code)


def transition_matrix(d, weight=None):
    """Row-normalised transition matrix (rows: from, columns: to)."""
    t = pd.crosstab(d["from"], d["to"],
                    values=None if weight is None else d[weight],
                    aggfunc=None if weight is None else "sum")
    t = t.reindex(index=STATES, columns=STATES).fillna(0)
    return t.div(t.sum(axis=1), axis=0)


counts = pd.crosstab(pairs["from"], pairs["to"]).reindex(index=STATES, columns=STATES)
P_unw = transition_matrix(pairs)
P = transition_matrix(pairs, weight="wq")
print("Number of person-pairs:")
print(counts)
pd.concat({"unweighted": P_unw.round(3), "weighted": P.round(3)}, axis=1)

In [ ]:
print(f"Pairs starting in unemployment: {counts.loc['U'].sum():,}")
print(f"p_UU (weighted) = {P.loc['U', 'U']:.3f}: of the urban unemployed this "
      f"quarter, {P.loc['U', 'U']:.0%} are still unemployed next quarter")
print(f"p_UE = {P.loc['U', 'E']:.3f}, p_UN = {P.loc['U', 'N']:.3f}")
print(f"p_EU = {P.loc['E', 'U']:.3f}: job loss into unemployment, per quarter")

### Who stays unemployed?

In [ ]:
pairs["group_sex"] = pairs["sex"].map({1: "Male", 2: "Female"})
pairs["age_group"] = pd.cut(pairs["age"], [14, 29, 59, 120],
                            labels=["15–29", "30–59", "60+"])


def key_flows(d):
    M = transition_matrix(d, "wq")
    return pd.Series({"p_UU": M.loc["U", "U"], "p_UE": M.loc["U", "E"],
                      "p_UN": M.loc["U", "N"], "p_EU": M.loc["E", "U"],
                      "p_NU": M.loc["N", "U"],
                      "n from U": int((d["from"] == "U").sum())})


by_group = pd.concat({
    "by sex": pairs.dropna(subset=["group_sex"]).groupby("group_sex").apply(key_flows),
    "by age": pairs.groupby("age_group", observed=True).apply(key_flows),
})
by_group.round(3)

## 27.6 Markov chains

In [ ]:
Pm = P.to_numpy()
for n in (1, 4, 8):
    Pn = np.linalg.matrix_power(Pm, n)
    print(f"P^{n}: starting unemployed, probability of E, U, N after {n} "
          f"quarter(s): {Pn[1].round(3)}")

### Expected duration

In [ ]:
durations = pd.Series({s: 1 / (1 - P.loc[s, s]) for s in STATES},
                      name="expected spell length, quarters")
print(durations.round(1))
print(f"Unemployment spell: {durations['U']:.1f} quarters "
      f"= {3 * durations['U']:.0f} months")
group_dur = (1 / (1 - by_group["p_UU"])).rename("expected U spell, quarters")
group_dur.round(1)

### The stationary distribution

In [ ]:
eigval, eigvec = np.linalg.eig(Pm.T)
k = np.argmin(np.abs(eigval - 1))              # the eigenvalue closest to 1
pi_eig = np.real(eigvec[:, k])
pi_eig = pi_eig / pi_eig.sum()                 # scale so that shares add to 1

pi = np.array([1.0, 0.0, 0.0])                 # start: everyone employed
for t in range(500):
    pi = pi @ Pm
print("eigenvalues of P:", np.real(eigval).round(4))
print("stationary distribution (eigenvector):", pi_eig.round(4))
print("stationary distribution (iteration):  ", pi.round(4))

In [ ]:
u_star = pi_eig[1] / (pi_eig[0] + pi_eig[1])
lfpr_star = pi_eig[0] + pi_eig[1]
actual_lfpr = adults.groupby("qtr", observed=True).apply(
    lambda d: labour_rates(d, "status", "wq")["LFPR"]).mean()
compare = pd.DataFrame({
    "steady state (Markov)": [100 * u_star, 100 * lfpr_star],
    "actual, 2023-24 average": [actual_ur, actual_lfpr]},
    index=["unemployment rate, %", "LFPR, %"])
compare.round(2)

### The two-state shortcut

In [ ]:
s_hat, f_hat = P.loc["E", "U"], P.loc["U", "E"]
print(f"s = {s_hat:.4f}, f = {f_hat:.4f}, u* = s/(s+f) = {s_hat / (s_hat + f_hat):.2%}")
print(f"three-state chain: u* = {u_star:.2%}")

### How fast does the chain forget? Mixing

In [ ]:
lam2 = sorted(np.abs(eigval))[-2]
print(f"|lambda_2| = {lam2:.4f}; half-life of a deviation: "
      f"{np.log(0.5) / np.log(lam2):.1f} quarters")

rows = []
for start in STATES:
    pi_t = np.array([1.0 if s == start else 0.0 for s in STATES])
    for t in range(41):
        rows.append({"start": start, "t": t,
                     "tv": 0.5 * np.abs(pi_t - pi_eig).sum(),
                     "u_share": pi_t[1]})
        pi_t = pi_t @ Pm
tv = pd.DataFrame(rows)
mix = tv[tv["tv"] < 0.25].groupby("start")["t"].min()
print("quarters until total variation distance < 0.25, by starting state:")
print(mix.to_dict())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colours = {"E": "#1f4e79", "U": "#c55a11", "N": "#9db7d5"}
names = {"E": "Start employed", "U": "Start unemployed",
         "N": "Start out of labour force"}
for s in STATES:
    d = tv[tv["start"] == s]
    axes[0].plot(d["t"], d["tv"], color=colours[s], lw=2, label=names[s])
axes[0].axhline(0.25, color="grey", ls=":", lw=1)
axes[0].set_title("Distance from the stationary distribution", loc="left",
                  fontsize=11)
axes[0].set_xlabel("Quarters")
axes[0].set_ylabel("Total variation distance")
axes[0].legend(frameon=False, fontsize=9)

d = tv[tv["start"] == "U"]
axes[1].plot(d["t"], 100 * d["u_share"], color="#c55a11", lw=2,
             label="P(unemployed) after t quarters")
axes[1].plot(d["t"], 100 * P.loc["U", "U"] ** d["t"], color="#1f4e79", lw=1.5,
             ls="--", label="P(still in the same spell) = $p_{UU}^t$")
axes[1].set_title("Starting from unemployment", loc="left", fontsize=11)
axes[1].set_xlabel("Quarters")
axes[1].set_ylabel("%")
axes[1].legend(frameon=False, fontsize=9)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Implied by the weighted quarterly transition matrix, "
         "urban persons 15+. Source: author's calculation from PLFS 2023–24 "
         "(NSO, MoSPI).", fontsize=8, color="grey")
plt.show()

## 27.7 Simulating work histories

In [ ]:
rng = np.random.default_rng(27)


def simulate(P, start, quarters, rng):
    """One simulated path of state indices (0=E, 1=U, 2=N)."""
    path = [start]
    for _ in range(quarters):
        path.append(rng.choice(3, p=P[path[-1]]))
    return path


n_people, T = 40, 24
starts = rng.choice(3, size=n_people, p=pi_eig)
starts[:8] = 1                                   # make sure some start unemployed
hist = np.array([simulate(Pm, s, T, rng) for s in starts])

fig, ax = plt.subplots(figsize=(10, 5))
from matplotlib.colors import ListedColormap
ax.imshow(hist, aspect="auto", interpolation="nearest",
          cmap=ListedColormap(["#1f4e79", "#c55a11", "#d9d9d9"]))
ax.hlines(np.arange(0.5, n_people - 1), -0.5, T + 0.5, color="white", lw=0.6)
ax.set_yticks(range(0, n_people, 5), range(1, n_people + 1, 5))
ax.set_xlabel("Quarter")
ax.set_ylabel("Simulated person")
ax.set_title("Simulated work histories: blue = employed, orange = unemployed, "
             "grey = out of LF", loc="left", fontsize=11)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "SIMULATED from the estimated quarterly transition "
         "matrix (urban persons 15+, PLFS 2023–24). Rows 1–8 start unemployed.",
         fontsize=8, color="grey")
plt.show()

In [ ]:
long = np.array([simulate(Pm, 1, 400, rng) for _ in range(200)])
spells = []
for row in long:
    run = 0
    for state in row:
        if state == 1:
            run += 1
        elif run > 0:
            spells.append(run)
            run = 0
spells = np.array(spells)
print(f"{len(spells):,} simulated spells; mean length {spells.mean():.2f} "
      f"quarters (theory {durations['U']:.2f})")
print(f"share of spells lasting a year or more (4+ quarters): "
      f"{(spells >= 4).mean():.1%} (theory {P.loc['U', 'U'] ** 3:.1%})")

## 27.8 Is unemployment really Markov?

In [ ]:
later2 = people[key + ["status"]].copy()
later2["q"] -= 2
later2["v"] -= 2
two = people[people["age"] >= 15].merge(later2, on=key, suffixes=("", "_2"))
two["from"] = two["status"].map(code)
two["to"] = two["status_2"].map(code)
P2_observed = transition_matrix(two, "wq")
P2_markov = pd.DataFrame(Pm @ Pm, index=STATES, columns=STATES)
print(f"{len(two):,} pairs two quarters apart")
pd.concat({"observed (v to v+2)": P2_observed.round(3),
           "Markov prediction P^2": P2_markov.round(3)}, axis=1)

In [ ]:
mid = later.rename(columns={"status": "status_mid"})[key + ["status_mid"]]
three = two.merge(mid, on=key)
three["mid"] = three["status_mid"].map(code)
hist_dep = (three[three["mid"] == "U"]
            .assign(still_u=lambda d: (d["to"] == "U").astype(float))
            .groupby("from")
            .apply(lambda d: pd.Series({
                "P(U next quarter | U now)": np.average(d["still_u"], weights=d["wq"]),
                "persons": len(d)})))
hist_dep.index.name = "status last quarter"
hist_dep.round(3)

In [ ]:
del later, later2, two, three, mid

## 27.9 Caveats: rotation, attrition and one year of data

## 27.10 Answering the question

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading